# Actividad Extra: Avistamientos UFO

## Objetivos
- Leer un archivo Excel con pandas y convertirlo a Spark DataFrame
- Limpiar y transformar datos con valores nulos
- Analizar distribucion geografica de avistamientos
- Explorar tendencias temporales
- Analizar formas reportadas y duracion de avistamientos

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import pandas as pd

spark = SparkSession.builder \
    .appName("actividad_avistamientos") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

---
## Ejercicio 1: Lectura del archivo Excel

Lee el archivo `ufo_sightings.xlsx` usando pandas y luego conviertelo a un Spark DataFrame. Explora las columnas y tipos de datos.

**Columnas del dataset:** Date_time, city, state/province, country, UFO_shape, length_of_encounter_seconds, described_duration_of_encounter, description, date_documented, latitude, longitude

In [ ]:
# =============================================================
# EJERCICIO 1: Lectura del archivo Excel
# =============================================================

# Leer con pandas
pdf = pd.read_excel("/home/jovyan/datos/ufo_sightings.xlsx")
print(f"Forma del DataFrame pandas: {pdf.shape}")
print(f"Columnas: {list(pdf.columns)}")
print(f"\nTipos de datos:")
print(pdf.dtypes)

# Convertir a Spark DataFrame
df = spark.createDataFrame(pdf)

# Explorar en Spark
df.printSchema()
df.show(5, truncate=50)
print(f"Total de registros: {df.count()}")

> **Nota docente:** Spark no puede leer archivos Excel nativamente (necesitaria el paquete `com.crealytics.spark.excel`), por lo que el flujo pandas -> Spark es el enfoque mas comun. Al usar `spark.createDataFrame(pdf)`, Spark infiere los tipos desde los tipos de pandas. Puede haber problemas de conversion si las columnas de pandas tienen tipos mixtos (ej: int + NaN se convierte a float). Los alumnos deben notar que la columna `state/province` contiene un `/` en el nombre, lo cual puede causar problemas en algunas operaciones de Spark.

---
## Ejercicio 2: Limpieza de datos

Analiza los valores nulos por columna y realiza una limpieza basica. Renombra la columna `state/province` a `state_province` para facilitar su uso.

In [ ]:
# =============================================================
# EJERCICIO 2: Limpieza de datos
# =============================================================

# Conteo de nulls por columna
print("Nulls por columna:")
df.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]
).show(truncate=False)

# Renombrar columna problematica
df = df.withColumnRenamed("state/province", "state_province")

# Eliminar filas donde Date_time es null
registros_antes = df.count()
df = df.na.drop(subset=["Date_time"])
registros_despues = df.count()
print(f"Registros eliminados por Date_time null: {registros_antes - registros_despues}")

# Rellenar nulls de UFO_shape con "Desconocida"
df = df.na.fill({"UFO_shape": "Desconocida"})

# Verificar
print(f"Registros despues de limpieza: {df.count()}")
df.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]
).show(truncate=False)

> **Nota docente:** La limpieza de datos es una etapa critica en cualquier analisis. `withColumnRenamed()` es necesario porque el nombre `state/province` con `/` puede causar ambiguedad en expresiones SQL. `na.drop(subset=[...])` elimina filas donde las columnas especificadas tienen null. `na.fill()` acepta un diccionario para rellenar nulls con valores especificos por columna. Es importante verificar el estado de los datos despues de cada operacion de limpieza para asegurarse de que las transformaciones fueron correctas. Los alumnos deben entender que eliminar filas vs rellenar nulls son decisiones que dependen del contexto del analisis.

---
## Ejercicio 3: Distribucion geografica

Cuenta los avistamientos por pais y muestra el top 10. Luego, para el pais con mas avistamientos, muestra el top 10 de estados/provincias.

In [ ]:
# =============================================================
# EJERCICIO 3: Distribucion geografica
# =============================================================

# Avistamientos por pais - top 10
df_pais = df.groupBy("country").count().orderBy("count", ascending=False)
print("Top 10 paises con mas avistamientos:")
df_pais.show(10)

# Grafico de barras
pdf_pais = df_pais.limit(10).toPandas()

plt.figure(figsize=(10, 5))
plt.bar(pdf_pais["country"].fillna("Desconocido"), pdf_pais["count"], color="#4CAF50")
plt.title("Top 10 Paises con Avistamientos UFO", fontsize=14)
plt.xlabel("Pais")
plt.ylabel("Numero de Avistamientos")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Top 10 estados del pais con mas avistamientos
pais_top = pdf_pais.iloc[0]["country"]
print(f"\nTop 10 estados/provincias en {pais_top}:")
df_estados = df.filter(F.col("country") == pais_top) \
    .groupBy("state_province").count() \
    .orderBy("count", ascending=False) \
    .limit(10)

df_estados.show()

# Grafico
pdf_estados = df_estados.toPandas()

plt.figure(figsize=(10, 5))
plt.barh(pdf_estados["state_province"].fillna("Desconocido"), pdf_estados["count"], color="#2196F3")
plt.title(f"Top 10 Estados/Provincias en {pais_top}", fontsize=14)
plt.xlabel("Numero de Avistamientos")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

> **Nota docente:** Estados Unidos domina ampliamente los avistamientos UFO, probablemente por sesgo de reporte y cultura popular. Dentro de EE.UU., California suele liderar. El uso de `fillna("Desconocido")` en pandas es necesario para que matplotlib no genere errores con valores None. Los alumnos pueden discutir las razones del sesgo geografico: poblacion, cultura de reporte, base de datos centrada en EE.UU., etc.

---
## Ejercicio 4: Tendencias temporales

Analiza las tendencias de avistamientos por anio, por mes y por dia de la semana usando la columna `Date_time`.

In [ ]:
# =============================================================
# EJERCICIO 4: Tendencias temporales
# =============================================================

# Extraer componentes de fecha
df_temp = df.withColumn("anio", F.year("Date_time")) \
    .withColumn("mes", F.month("Date_time")) \
    .withColumn("dia_semana", F.dayofweek("Date_time"))

# --- Por anio ---
df_anio = df_temp.groupBy("anio").count().orderBy("anio")
pdf_anio = df_anio.toPandas()

plt.figure(figsize=(12, 5))
plt.plot(pdf_anio["anio"], pdf_anio["count"], linewidth=1.5, color="#4CAF50")
plt.title("Avistamientos UFO por Anio", fontsize=14)
plt.xlabel("Anio")
plt.ylabel("Cantidad")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- Por mes ---
df_mes = df_temp.groupBy("mes").count().orderBy("mes")
pdf_mes = df_mes.toPandas()
meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

plt.figure(figsize=(10, 5))
plt.bar(range(1, len(pdf_mes) + 1), pdf_mes["count"], color="#FF9800")
plt.xticks(range(1, 13), meses)
plt.title("Avistamientos UFO por Mes", fontsize=14)
plt.xlabel("Mes")
plt.ylabel("Cantidad")
plt.tight_layout()
plt.show()

# --- Por dia de semana ---
df_dia = df_temp.groupBy("dia_semana").count().orderBy("dia_semana")
pdf_dia = df_dia.toPandas()
dias = ["Dom", "Lun", "Mar", "Mie", "Jue", "Vie", "Sab"]

plt.figure(figsize=(8, 5))
plt.bar(range(1, len(pdf_dia) + 1), pdf_dia["count"], color="#9C27B0")
plt.xticks(range(1, 8), dias)
plt.title("Avistamientos UFO por Dia de la Semana", fontsize=14)
plt.xlabel("Dia")
plt.ylabel("Cantidad")
plt.tight_layout()
plt.show()

> **Nota docente:** Las funciones `F.year()`, `F.month()` y `F.dayofweek()` extraen componentes de una columna timestamp/date. El grafico por anio muestra un aumento sostenido de reportes, posiblemente relacionado con la difusion de internet y redes sociales. El grafico por mes puede revelar estacionalidad (mas avistamientos en verano del hemisferio norte, cuando hay mas horas de oscuridad tardia y actividades al aire libre). Los fines de semana pueden tener mas avistamientos porque la gente tiene mas tiempo libre para observar el cielo.

---
## Ejercicio 5: Duracion de los avistamientos

Analiza la columna `length_of_encounter_seconds` con estadisticas descriptivas. Categoriza la duracion en rangos y muestra la distribucion.

In [ ]:
# =============================================================
# EJERCICIO 5: Duracion de los avistamientos
# =============================================================

# Estadisticas descriptivas
print("Estadisticas de duracion (segundos):")
df.select("length_of_encounter_seconds").describe().show()

# Crear rangos de duracion
df_duracion = df.withColumn("rango_duracion",
    F.when(F.col("length_of_encounter_seconds") < 60, "Muy corto (<1 min)")
    .when((F.col("length_of_encounter_seconds") >= 60) & (F.col("length_of_encounter_seconds") < 300), "Corto (1-5 min)")
    .when((F.col("length_of_encounter_seconds") >= 300) & (F.col("length_of_encounter_seconds") < 1800), "Medio (5-30 min)")
    .when((F.col("length_of_encounter_seconds") >= 1800) & (F.col("length_of_encounter_seconds") < 3600), "Largo (30-60 min)")
    .otherwise("Muy largo (>1 hora)")
)

# Contar por rango
df_rangos = df_duracion.groupBy("rango_duracion").count() \
    .orderBy("count", ascending=False)

df_rangos.show(truncate=False)

# Grafico de barras
pdf_rangos = df_rangos.toPandas()

plt.figure(figsize=(10, 5))
plt.bar(pdf_rangos["rango_duracion"], pdf_rangos["count"], color="#FF5722")
plt.title("Distribucion de Duracion de Avistamientos", fontsize=14)
plt.xlabel("Rango de Duracion")
plt.ylabel("Cantidad")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

> **Nota docente:** La categorizacion de variables continuas en rangos es un patron muy comun en analisis exploratorio. La columna `length_of_encounter_seconds` puede tener valores extremos (outliers) que distorsionan la media pero no la mediana. Los alumnos deben notar que `describe()` solo muestra media y desviacion estandar, no la mediana; para obtenerla se usaria `F.percentile_approx("col", 0.5)`. La mayoria de avistamientos suelen ser de duracion corta. El uso de rangos legibles en espanol facilita la interpretacion del grafico.

---
## Ejercicio 6: Formas reportadas de UFOs

Analiza la columna `UFO_shape` para ver cuales son las formas mas reportadas. Crea un grafico de barras con el top 15.

In [ ]:
# =============================================================
# EJERCICIO 6: Formas reportadas
# =============================================================

# Contar por forma
df_formas = df.groupBy("UFO_shape").count() \
    .orderBy("count", ascending=False) \
    .limit(15)

df_formas.show(truncate=False)

# Grafico de barras horizontales
pdf_formas = df_formas.toPandas()

plt.figure(figsize=(10, 7))
plt.barh(pdf_formas["UFO_shape"], pdf_formas["count"], color="#4CAF50")
plt.title("Top 15 Formas de UFO Reportadas", fontsize=14)
plt.xlabel("Numero de Avistamientos")
plt.ylabel("Forma")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

> **Nota docente:** Las formas mas comunes suelen ser "light" (luz), "triangle" (triangulo), "circle" (circulo) y "fireball" (bola de fuego). La categoria "Desconocida" que rellenamos en la limpieza tambien aparece. Los alumnos pueden discutir por que las luces son la forma mas comun (son lo mas visible en el cielo nocturno y lo mas ambiguo de identificar). Para un analisis mas avanzado, se podria cruzar la forma con la duracion o la ubicacion geografica para buscar patrones.

---
## Resumen

En esta actividad aprendimos:

1. **Lectura de Excel** con pandas y conversion a Spark DataFrame
2. **Limpieza de datos** con manejo de nulls, renombrado de columnas
3. **Analisis geografico** con agrupaciones por pais y estado
4. **Tendencias temporales** extrayendo componentes de fecha
5. **Categorizacion** de variables continuas en rangos con `F.when()`
6. **Analisis de frecuencia** de variables categoricas

In [ ]:
# Detener SparkSession
spark.stop()
print("SparkSession detenida correctamente.")